In [1]:
# Import pandas library to work with datasets.
import pandas as pd

In [2]:
# Install openpyxl to enable reading Excels.
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [3]:
sos_total_og = pd.read_excel("../data_storage/raw_data/voter_data/2024_total_gen_election.xlsx")
sos_gender_og = pd.read_excel("../data_storage/raw_data/voter_data/2024_gender_gen_election.xlsx")
sos_age_og = pd.read_excel("../data_storage/raw_data/voter_data/2024_age_gen_election.xlsx")
sos_race_og = pd.read_excel("../data_storage/raw_data/voter_data/2024_race_gen_election.xlsx")
rdh_og = pd.read_csv("../data_storage/raw_data/voter_data/AL_l2_2024_gen_stats_2020county.csv")
alvr_og = pd.read_excel("../data_storage/raw_data/voter_data/2024_votereg_data_AlSoS.xlsx", sheet_name="October", header=None, engine="openpyxl")

In [4]:
sos_total = sos_total_og.copy()
sos_gender = sos_gender_og.copy()
sos_age = sos_age_og.copy()
sos_race = sos_race_og.copy()
rdh = rdh_og.copy()
alvr = alvr_og.copy()

In [5]:
#SoS and RDH have different spacing/naming so needed to strip/rename columns. fips_code is a more standardized naming.
sos_total.columns = sos_total.columns.str.strip().str.replace("\n", " ")
sos_gender.columns = sos_gender.columns.str.strip().str.replace("\n", " ")
sos_age.columns = sos_age.columns.str.strip().str.replace("\n", " ")
sos_race.columns = sos_race.columns.str.strip().str.replace("\n", " ")

sos_total["County"] = sos_total["County"].str.strip().str.title()
sos_gender["County"] = sos_gender["County"].str.strip().str.title()
sos_age["County"] = sos_age["County"].str.strip().str.title()
sos_race["County"] = sos_race["County"].str.strip().str.title()

rdh = rdh.rename(columns={"countyname": "County", "countyfips": "fips_code"})
rdh["County"] = rdh["County"].str.strip().str.title()

#ALVR from SoS has a broken two-row header so manually set clean column names
alvr.columns = ['County','Total_AI','Active_Asian','Active_AmInd','Active_Black','Active_Fed','Active_Hispanic','Active_Korean','Active_White','Active_Other','Active_NotId','Active_Total','Inactive_Asian','Inactive_AmInd','Inactive_Black','Inactive_Fed','Inactive_Hispanic','Inactive_Korean','Inactive_White','Inactive_Other','Inactive_NotId','Inactive_Total']

#Drops rows 0 and 1 which contained the broken header labels then resets the index so rows start at 0
alvr = alvr.iloc[2:].reset_index(drop=True)
alvr["County"] = alvr["County"].astype(str).str.strip().str.title()

In [6]:
#Filters the 17 counties for Alabama BB
BLACK_BELT_COUNTIES = ["Sumter", "Choctaw", "Greene", "Hale", "Marengo", "Perry", "Dallas", "Wilcox", "Lowndes", "Butler", "Crenshaw", "Montgomery", "Pike", "Bullock", "Macon", "Barbour", "Russell"]

sos_total = sos_total[sos_total["County"].isin(BLACK_BELT_COUNTIES)].copy()
sos_gender = sos_gender[sos_gender["County"].isin(BLACK_BELT_COUNTIES)].copy()
sos_age = sos_age[sos_age["County"].isin(BLACK_BELT_COUNTIES)].copy()
sos_race = sos_race[sos_race["County"].isin(BLACK_BELT_COUNTIES)].copy()
rdh_bb_filtered = rdh[rdh["County"].isin(BLACK_BELT_COUNTIES)].copy()
alvr_bb_filtered = alvr[alvr["County"].isin(BLACK_BELT_COUNTIES)].copy()

In [7]:
#Slim each SoS breakdown file before merging to avoid duplicate Total Ballots Cast columns
gender_slim = sos_gender[["County", "Total Female", "Total Male", "Total Not Identified"]].rename(columns={"Total Not Identified": "ballots_not_identified_gender"})
age_slim = sos_age[["County", "Age 18-29", "Age 30-39", "Age 40-49", "Age 50-59", "Age 60-69", "Age 70-79", "Age 80-89", "Age 90-99", "Age 100+"]]
race_slim = sos_race[["County", "Total Black (B)", "Total White (W)", "Total Hispanic (H)"]]

In [8]:
#Inner join to merge all four SoS files into one participation dataframe, add FIPS code from RDH and active registered by race from ALVR
participation = sos_total.merge(gender_slim, on="County").merge(age_slim, on="County").merge(race_slim, on="County")
participation = participation.merge(rdh_bb_filtered[["County", "fips_code"]], on="County")
participation = participation.merge(alvr_bb_filtered[["County", "Active_Black", "Active_White", "Active_Hispanic"]], on="County")

In [9]:
#Combine SoS age buckets 80-89, 90-99, 100+ into a single 80+ column
participation["ballots_age_80_plus"]=participation["Age 80-89"] + participation["Age 90-99"] + participation["Age 100+"]

In [10]:
#Calculate race-based turnout rates using active registered voters from ALVR October snapshot
#Force float conversion on both sides so rounding applies correctly since ALVR columns may be object type
participation["turnout_rate_black"] = (participation["Total Black (B)"].astype(float) / participation["Active_Black"].astype(float)).round(4)
participation["turnout_rate_white"] = (participation["Total White (W)"].astype(float) / participation["Active_White"].astype(float)).round(4)
participation["turnout_rate_latinx"] = (participation["Total Hispanic (H)"].astype(float) / participation["Active_Hispanic"].astype(float)).round(4)

In [12]:
#Rename all columns to follow naming convention: lowercase with underscores
participation = participation.rename(columns={"County": "county_name","Registered Voters": "registered_voters","Total Ballots Cast": "total_ballots_cast",
                                              "Absentee Ballots": "absentee_ballots","Turnout as a Percentage of Registered Voters": "turnout_rate", "Absentee as Percentage of Total Ballots": "absentee_rate","Total Female": "ballots_female","Total Male": "ballots_male","Age 18-29": "ballots_age_18_29","Age 30-39": "ballots_age_30_39","Age 40-49": "ballots_age_40_49","Age 50-59": "ballots_age_50_59","Age 60-69": "ballots_age_60_69","Age 70-79": "ballots_age_70_79","Total Black (B)": "ballots_black","Total White (W)": "ballots_white", "Total Hispanic (H)": "ballots_latinx"})

In [13]:
#Keep only final output columns in the order SPLC will read them
final_columns = ["county_name", "fips_code", "registered_voters", "total_ballots_cast","absentee_ballots", "turnout_rate", "absentee_rate","ballots_female", "ballots_male", "ballots_not_identified_gender",
    "ballots_age_18_29", "ballots_age_30_39", "ballots_age_40_49", "ballots_age_50_59","ballots_age_60_69", "ballots_age_70_79", "ballots_age_80_plus", "ballots_black", "ballots_white", "ballots_latinx",
    "turnout_rate_black", "turnout_rate_white", "turnout_rate_latinx"]

participation = participation[final_columns].sort_values("county_name").reset_index(drop=True)

In [15]:
#Display final cleaned participation dataset
participation
# Export to cleaned data folder
participation.to_csv("../data_storage/clean_data/voter_turnout_2024_general.csv", index=False)